In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Binary ESI 1 XGBoost Classifier with Factor-Controlled Majority Undersampling (`models/xgboost_raw_esi1_extreme.ipynb`)

This notebook trains a **Binary XGBoost Classifier** for **ESI 1 vs Not ESI 1** combining **29 Predictor Features** (3 baseline raw features + 10 binary vital anomaly flags + 16 continuous vital delta & range features) with **Factor-Controlled Majority Class Undersampling**:

### System Architecture & Workflow
1. **Predictor Feature Selection (29 Predictors)**:
   - **Baseline Raw Features (3)**: `age`, `gender`, `cc_breathingdifficulty`.
   - **10 Binary Vital Anomaly Flags**: `is_dyspnea_total`, `is_dyspnea_moderate`, `is_bradypnea`, `is_tachypnea`, `is_hypotension`, `is_hypertension`, `is_bradycardia_total`, `is_bradycardia_moderate`, `is_tachycardia_total`, `is_tachycardia_moderate`.
   - **16 Continuous Vital Delta & Range Features**: `hr_mean_to_last`, `sbp_mean_to_last`, `spo2_mean_to_last`, `rr_mean_to_last`, `hr_range`, `rr_range`, `spo2_range`, `sbp_range`, `hr_last_to_min`, `rr_last_to_min`, `spo2_last_to_min`, `sbp_last_to_min`, `hr_last_to_max`, `rr_last_to_max`, `spo2_last_to_max`, `sbp_last_to_max`.
2. **Stratified Data Partitioning**: Splits dataset into Train (70%), Validation (15%), and Test (15%) splits prior to scaling to prevent data leakage.
3. **Factor-Controlled Majority Undersampling**: Controls majority class sampling via `undersample_ratio` parameter (e.g. `undersample_ratio = 1.0` retains $1.0 	imes N_{\text{esi1}}$ majority cases), balancing the training set while preserving natural validation & test distributions.
4. **Standard Binary XGBoost Gradient Boosting**: Fits binary objective decision trees (`objective = "binary:logistic"`, `eval_metric = "logloss"`) via `xgb.DMatrix` and `xgb.train()`.
5. **Comprehensive Benchmarking Across Splits**: Evaluates Validation and Test performance with confusion matrices, Accuracy, Precision, Recall, F1 Score, ROC-AUC, and **MCC Score**.
6. **Reports & Artifacts**:
   - **Diagnostic Plots**: Metrics bar chart (`plots/xgboost_raw_esi1_metrics_barchart.png`).
   - **CSV Reports**: `reports/xgboost_raw_esi1_val_report.csv`, `reports/xgboost_raw_esi1_test_report.csv`.
   - **Model Export**: Saved to `deploy/xgboost_raw_esi1_extreme_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(xgboost)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Construct 29 Predictor Features
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last")
p_max    <- get_vec("pulse_max")
p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last")
s_max    <- get_vec("sbp_max")
s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last")
o2_max   <- get_vec("spo2_max")
o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last")
r_max    <- get_vec("resp_max")
r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr")
t_sbp    <- get_vec("triage_vital_sbp")
t_o2     <- get_vec("triage_vital_o2")
t_rr     <- get_vec("triage_vital_rr")
# Construct 29 Predictor Features
df_full <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0),
  
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
raw_esi <- as.character(raw_df[[target_col]])
df_full$target_layer1 <- factor(ifelse(raw_esi == "1", "1", "not_1"), levels = c("1", "not_1"))
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA raw features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df_full), nrow(df_full)))
cat(sprintf("Full Feature Dataset Ready (Pre-Partitioning): %d total rows x %d cols\n", nrow(df_full), ncol(df_full)))
cat("Predictor Features Included (29 Total):\n")
print(setdiff(names(df_full), "target_layer1"))
cat("\nNatural Binary Target Distribution ('1' vs 'not_1'):\n")
print(table(df_full$target_layer1))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Partitioning & Factor-Controlled Majority Undersampling
# ---------------------------------------------------------
set.seed(config$training$random_state)
test_size <- config$training$test_size
val_size  <- config$training$val_size
in_train_val <- createDataPartition(df_full$target_layer1, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
val_prop <- val_size / (1 - test_size)
in_train <- createDataPartition(train_val_df$target_layer1, p = 1 - val_prop, list = FALSE)
train_df <- train_val_df[in_train, ]
val_df   <- train_val_df[-in_train, ]
# FACTOR-CONTROLLED MAJORITY UNDERSAMPLING PARAMETER
undersample_ratio <- 1.0  # 1.0 means 1:1 ratio (N_not1 = 1.0 * N_esi1)
train_esi1_idx  <- which(train_df$target_layer1 == "1")
train_not1_idx  <- which(train_df$target_layer1 == "not_1")
n_esi1 <- length(train_esi1_idx)
n_not1_target <- round(n_esi1 * undersample_ratio)
if (n_not1_target < length(train_not1_idx) && n_not1_target > 0) {
  sampled_not1_idx <- sample(train_not1_idx, size = n_not1_target, replace = FALSE)
  keep_train_idx   <- c(train_esi1_idx, sampled_not1_idx)
  train_df         <- train_df[keep_train_idx, ]
}
cat("=== Stratified Partitioning & Majority Undersampling Summary ===\n")
cat(sprintf("Undersampling Control Ratio : %.2f (N_not1_target = %.2f * N_esi1)\n", undersample_ratio, undersample_ratio))
cat(sprintf("Train Set (Undersampled)   : %d rows (ESI 1: %d, Not ESI 1: %d)\n",
            nrow(train_df), sum(train_df$target_layer1 == "1"), sum(train_df$target_layer1 == "not_1")))
cat(sprintf("Validation Set (Natural)   : %d rows (ESI 1: %d, Not ESI 1: %d)\n",
            nrow(val_df), sum(val_df$target_layer1 == "1"), sum(val_df$target_layer1 == "not_1")))
cat(sprintf("Holdout Test Set (Natural) : %d rows\n\n", nrow(test_df)))
# Standardize continuous features
binary_cols <- c("gender", "cc_breathingdifficulty",
                 "is_dyspnea_total", "is_dyspnea_moderate", "is_bradypnea", "is_tachypnea",
                 "is_hypotension", "is_hypertension", "is_bradycardia_total", "is_bradycardia_moderate",
                 "is_tachycardia_total", "is_tachycardia_moderate")
cont_cols <- setdiff(names(train_df), c(binary_cols, "target_layer1"))
preproc  <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train Binary XGBoost Model
# ---------------------------------------------------------
set.seed(config$training$random_state)
train_x <- as.matrix(train_df[, setdiff(names(train_df), "target_layer1")])
train_y <- ifelse(train_df$target_layer1 == "1", 1, 0)
val_x   <- as.matrix(val_df[, setdiff(names(val_df), "target_layer1")])
val_y   <- ifelse(val_df$target_layer1 == "1", 1, 0)
test_x  <- as.matrix(test_df[, setdiff(names(test_df), "target_layer1")])
test_y  <- ifelse(test_df$target_layer1 == "1", 1, 0)
dtrain <- xgb.DMatrix(data = train_x, label = train_y)
dval   <- xgb.DMatrix(data = val_x,   label = val_y)
dtest  <- xgb.DMatrix(data = test_x,  label = test_y)
xgb_params <- list(
  objective        = "binary:logistic",
  eval_metric      = "logloss",
  eta              = 0.05,
  max_depth        = 6,
  subsample        = 0.8,
  colsample_bytree = 0.8
)
xgb_model <- xgb.train(
  params        = xgb_params,
  data          = dtrain,
  nrounds       = 100,
  watchlist     = list(train = dtrain, val = dval),
  early_stopping_rounds = 30,
  print_every_n = 50
)
cat(sprintf("XGBoost Training Complete. Best Iteration: %d\n", xgb_model$best_iteration))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Benchmark Evaluation Across Splits (Validation & Holdout Test with MCC)
# ---------------------------------------------------------
calc_mcc <- function(tp, fp, fn, tn) {
  num <- (tp * tn) - (fp * fn)
  denom <- sqrt(as.numeric(tp + fp) * as.numeric(tp + fn) * as.numeric(tn + fp) * as.numeric(tn + fn))
  if (is.na(denom) || denom == 0) 0 else num / denom
}
eval_split <- function(dmat, actual_y, split_name) {
  probs <- predict(xgb_model, dmat)
  preds <- ifelse(probs >= 0.5, 1, 0)
  
  pred_fac <- factor(ifelse(preds == 1, "1", "not_1"), levels = c("1", "not_1"))
  act_fac  <- factor(ifelse(actual_y == 1, "1", "not_1"), levels = c("1", "not_1"))
  
  cm  <- confusionMatrix(pred_fac, act_fac, positive = "1")
  acc <- as.numeric(cm$overall["Accuracy"])
  prec <- as.numeric(cm$byClass["Pos Pred Value"])
  rec  <- as.numeric(cm$byClass["Sensitivity"])
  prec <- ifelse(is.na(prec), 0, prec)
  rec  <- ifelse(is.na(rec), 0, rec)
  f1   <- ifelse((prec + rec) > 0, 2 * (prec * rec) / (prec + rec), 0)
  
  r_obj   <- tryCatch(pROC::roc(actual_y, probs), error = function(e) NULL)
  roc_auc <- if (!is.null(r_obj)) as.numeric(r_obj$auc) else NA
  
  tp <- sum(preds == 1 & actual_y == 1)
  tn <- sum(preds == 0 & actual_y == 0)
  fp <- sum(preds == 1 & actual_y == 0)
  fn <- sum(preds == 0 & actual_y == 1)
  mcc <- calc_mcc(tp, fp, fn, tn)
  
  cat(sprintf("=== Benchmark Evaluation: %s Split ===\n", split_name))
  cat(sprintf("  Accuracy   : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Precision  : %.4f\n", prec))
  cat(sprintf("  Recall     : %.4f\n", rec))
  cat(sprintf("  F1 Score   : %.4f\n", f1))
  cat(sprintf("  ROC-AUC    : %.4f\n", roc_auc))
  cat(sprintf("  MCC Score  : %.4f\n", mcc))
  cat("  Confusion Matrix:\n")
  print(cm$table)
  cat("\n")
  
  return(list(acc = acc, prec = prec, rec = rec, f1 = f1, roc_auc = roc_auc, mcc = mcc, cm = cm$table))
}
val_res  <- eval_split(dval,  val_y,  "Validation")
test_res <- eval_split(dtest, test_y, "Holdout Test")
# Write CSV Reports
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
val_df_report  <- data.frame(Metric = c("Accuracy", "Precision", "Recall", "F1_Score", "ROC_AUC", "MCC_Score"), Score = round(c(val_res$acc, val_res$prec, val_res$rec, val_res$f1, val_res$roc_auc, val_res$mcc), 4))
test_df_report <- data.frame(Metric = c("Accuracy", "Precision", "Recall", "F1_Score", "ROC_AUC", "MCC_Score"), Score = round(c(test_res$acc, test_res$prec, test_res$rec, test_res$f1, test_res$roc_auc, test_res$mcc), 4))
write.csv(val_df_report,  file = file.path(reports_dir, "xgboost_raw_esi1_val_report.csv"),  row.names = FALSE)
write.csv(test_df_report, file = file.path(reports_dir, "xgboost_raw_esi1_test_report.csv"), row.names = FALSE)
cat("CSV Reports written to reports/xgboost_raw_esi1_val_report.csv and reports/xgboost_raw_esi1_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Diagnostic Plots & Model Export
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
metrics_summary <- data.frame(
  Metric = factor(c("Val_Accuracy", "Val_Precision", "Val_Recall", "Val_F1", "Val_ROC_AUC", "Val_MCC",
                    "Test_Accuracy", "Test_Precision", "Test_Recall", "Test_F1", "Test_ROC_AUC", "Test_MCC"),
                  levels = c("Val_Accuracy", "Val_Precision", "Val_Recall", "Val_F1", "Val_ROC_AUC", "Val_MCC",
                             "Test_Accuracy", "Test_Precision", "Test_Recall", "Test_F1", "Test_ROC_AUC", "Test_MCC")),
  Score  = c(val_res$acc, val_res$prec, val_res$rec, val_res$f1, val_res$roc_auc, val_res$mcc,
             test_res$acc, test_res$prec, test_res$rec, test_res$f1, test_res$roc_auc, test_res$mcc)
)
p_bar <- ggplot(metrics_summary, aes(x = Metric, y = Score, fill = Metric)) +
  geom_bar(stat = "identity", width = 0.5) +
  geom_text(aes(label = sprintf("%.3f", Score)), vjust = -0.3, size = 3.2, fontface = "bold") +
  theme_minimal() +
  scale_fill_brewer(palette = "Set2") +
  labs(title = "Binary ESI 1 XGBoost Performance (Undersampled)",
       subtitle = sprintf("Majority Undersampling Control Ratio = %.2f", undersample_ratio),
       y = "Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13, hjust = 0.5),
        plot.subtitle = element_text(size = 10, hjust = 0.5),
        axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "none")
ggsave(file.path(plots_dir, "xgboost_raw_esi1_metrics_barchart.png"), plot = p_bar, width = 10, height = 5, dpi = 300)
cat("Metrics Comparison Bar Chart saved to: plots/xgboost_raw_esi1_metrics_barchart.png\n")
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
model_path <- file.path(deploy_dir, "xgboost_raw_esi1_extreme_model.rds")
saveRDS(list(model = xgb_model, preproc = preproc, undersample_ratio = undersample_ratio), file = model_path)
cat("Binary ESI 1 XGBoost Model saved to:", model_path, "\n")
p_bar